# MAS-SHT v10.2 — Experiment Harness

Drives all comparative experiments end-to-end on Google Colab. Six cells, top to bottom:
1. **Setup** — installs deps, mounts Drive, clones repo, loads secrets.
2. **Dataset** — GSM8K test split, configurable sampling.
3. **Runner** — runs all configured baselines + MAS-SHT presets, with checkpointing.
4. **Aggregation** — merges per-system CSVs into one DataFrame.
5. **Statistics** — McNemar tests, comparison table.
6. **Plots** — three-panel figure for the thesis.

Edit `CONFIG` in cell 3 to control which systems run, how many problems, and the seed.

## Cell 1 — Setup

In [ ]:
# === Cell 1 — Setup ============================================================
# Installs all deps, mounts Google Drive, clones the project, loads secrets.
# Safe to re-run: pip skips already-installed pkgs, Drive mount is idempotent.

import os, sys, subprocess

DEPS = [
    'openai>=1.40.0', 'google-generativeai', 'sympy', 'statsmodels',
    'scipy', 'pandas', 'matplotlib', 'tqdm', 'datasets',
    'transformers>=4.44.0', 'accelerate', 'huggingface_hub',
    'python-dotenv', 'tabulate', 'requests',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)

# Mount Google Drive (results + checkpoints persist here).
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — falling back to local filesystem.')

# Clone the project repo (or pull if already present).
REPO_URL = 'https://github.com/marios4371/llm_thesis.git'
REPO_DIR = '/content/MAS_LLM_Thesis' if IN_COLAB else os.path.expanduser('~/MAS_LLM_Thesis')

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    else:
        subprocess.call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Load API keys from Colab Secrets. Missing keys -> warn, do not crash.
def _get_secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

for k in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'HF_API_KEY', 'TOGETHER_API_KEY']:
    v = _get_secret(k)
    if v:
        os.environ[k] = v
    else:
        print(f'WARNING: secret {k} not set — presets that need it will fail.')

# Output directories.
ROOT = '/content/drive/MyDrive/MAS_SHT' if IN_COLAB else os.path.expanduser('~/MAS_SHT')
RESULTS_DIR = f'{ROOT}/results'
CHECKPOINT_DIR = f'{ROOT}/checkpoints'
ARTIFACTS_DIR = f'{ROOT}/artifacts'
for d in [RESULTS_DIR, CHECKPOINT_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Repo: {REPO_DIR}')
print(f'Results: {RESULTS_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## Cell 2 — Dataset

In [ ]:
# === Cell 2 — Dataset ==========================================================
# Loads GSM8K test split. Three sampling modes:
#   {'mode': 'full'}                         all 1319 problems
#   {'mode': 'random',     'n': 100, 'seed': 42}
#   {'mode': 'stratified', 'n': 100, 'seed': 42}   word-count terciles

import re, random
import pandas as pd
from datasets import load_dataset

DATASET_CONFIG = {'mode': 'random', 'n': 50, 'seed': 42}  # safe default for free-tier

def _parse_gold(answer_text: str):
    if '####' in answer_text:
        tail = answer_text.split('####')[-1].strip()
    else:
        tail = answer_text
    nums = re.findall(r'-?\d+(?:,\d+)*(?:\.\d+)?', tail)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(',', ''))
    except ValueError:
        return None

def load_problems(cfg):
    ds = load_dataset('openai/gsm8k', 'main', split='test')
    rng = random.Random(cfg.get('seed', 42))
    items = []
    for i, row in enumerate(ds):
        gold = _parse_gold(row['answer'])
        if gold is None:
            continue
        items.append({
            'problem_id': f'gsm8k_test_{i}',
            'question': row['question'],
            'gold_raw': row['answer'],
            'gold_answer': gold,
            'word_count': len(row['question'].split()),
        })
    mode = cfg.get('mode', 'random')
    if mode == 'full':
        return items
    if mode == 'random':
        rng.shuffle(items)
        return items[: cfg['n']]
    if mode == 'stratified':
        items.sort(key=lambda x: x['word_count'])
        n = len(items)
        thirds = [items[: n // 3], items[n // 3 : 2 * n // 3], items[2 * n // 3 :]]
        per_band = cfg['n'] // 3
        out = []
        for band in thirds:
            rng.shuffle(band)
            out += band[:per_band]
        rng.shuffle(out)
        return out
    raise ValueError(f'Unknown sampling mode: {mode}')

PROBLEMS = load_problems(DATASET_CONFIG)
print(f'Loaded {len(PROBLEMS)} problems (mode={DATASET_CONFIG["mode"]}).')
print('Example:', PROBLEMS[0]['question'][:120], '...')

## Cell 3 — Experiment Runner

In [ ]:
# === Cell 3 — Experiment Runner ================================================
# Iterates every configured (system, preset) over PROBLEMS with checkpointing.
# A timeout in Colab does not lose progress: re-running this cell resumes from
# the latest checkpoint per (system, preset).

import os, time, pickle, traceback
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

import Mas_solver
from Mas_solver import (
    QualityAwarePipeline, UnifiedLLMClient, AgentRole,
    HETEROGENEOUS_PRESETS, token_budget, _extract_last_number,
)

# ------------------------------------------------------------------
# CONFIG — edit me to control the experiment.
# ------------------------------------------------------------------
CONFIG = {
    # Which baselines to run.
    # B1=direct answer, B2=chain-of-thought, B3=self-consistency(5), B4=baseline_only
    'baselines_to_run': ['b1_direct', 'b2_cot', 'b3_sc5', 'b4_baseline_only'],
    # Provider+model used by simple baselines (B1-B4).
    'baseline_client': {'provider': 'groq', 'model': 'llama-3.3-70b-versatile'},
    # MAS variants: (system_name, preset, enable_siv, enable_sht)
    'mas_variants': [
        ('mas_no_siv',   'homogeneous_groq', False, True),   # B5
        ('mas_no_sht',   'homogeneous_groq', True,  False),  # B6
        ('mas_sht_full', 'homogeneous_groq', True,  True),   # B7 (reference)
        # Small models — uncomment to enable:
        # ('mas_sht_tiny',   'tiny_math_homogeneous', True, True),
        # ('mas_sht_qwen7b', 'qwen_math_7b',          True, True),
    ],
    'inter_problem_delay': 3.0,
    'checkpoint_every': 5,
}

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

def _ckpt_path(system_name):
    return os.path.join(CHECKPOINT_DIR, f'{system_name}.pkl')

def _csv_path(system_name):
    return os.path.join(RESULTS_DIR, f'{system_name}_{TIMESTAMP}.csv')

def _load_ckpt(system_name):
    p = _ckpt_path(system_name)
    if os.path.isfile(p):
        try:
            with open(p, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f'  checkpoint load failed for {system_name}: {e}')
    return []

def _save_ckpt(system_name, rows):
    with open(_ckpt_path(system_name), 'wb') as f:
        pickle.dump(rows, f)

def _is_correct(pred, gold):
    if pred is None or gold is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 1e-3
    except (TypeError, ValueError):
        return False

def _budget_blocked(err: str):
    return ('budget_exceeded' in err.lower() or 'rate_limit_daily' in err.lower())

# ------------------------------------------------------------------
# Baseline functions (B1-B4) — inline implementations
# ------------------------------------------------------------------
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class BaselineResult:
    answer: Optional[float]
    raw: str
    num_llm_calls: int
    tokens_estimated: int
    time_s: float
    error_type: str = ''

def _call(client, prompt, max_tokens=512):
    return client.call_model(prompt, max_tokens=max_tokens, temperature=0.0)

def direct_answer(client, problem: str) -> BaselineResult:
    t0 = time.time()
    prompt = f'Solve this math problem. Give only the final numeric answer.\n\n{problem}'
    try:
        raw = _call(client, prompt)
        ans = _extract_last_number(raw)
        return BaselineResult(answer=ans, raw=raw, num_llm_calls=1,
                              tokens_estimated=600, time_s=time.time()-t0)
    except Exception as e:
        return BaselineResult(answer=None, raw=str(e), num_llm_calls=1,
                              tokens_estimated=0, time_s=time.time()-t0, error_type='exception')

def chain_of_thought(client, problem: str) -> BaselineResult:
    t0 = time.time()
    prompt = (f'Solve this math problem step by step. '
              f'At the end write "Answer: <number>".\n\n{problem}')
    try:
        raw = _call(client, prompt, max_tokens=800)
        ans = _extract_last_number(raw)
        return BaselineResult(answer=ans, raw=raw, num_llm_calls=1,
                              tokens_estimated=900, time_s=time.time()-t0)
    except Exception as e:
        return BaselineResult(answer=None, raw=str(e), num_llm_calls=1,
                              tokens_estimated=0, time_s=time.time()-t0, error_type='exception')

def self_consistency(client, problem: str, n: int = 5) -> BaselineResult:
    t0 = time.time()
    prompt = (f'Solve this math problem step by step. '
              f'At the end write "Answer: <number>".\n\n{problem}')
    answers = []
    raw_all = []
    for _ in range(n):
        try:
            raw = client.call_model(prompt, max_tokens=800, temperature=0.7)
            a = _extract_last_number(raw)
            if a is not None:
                answers.append(a)
            raw_all.append(raw)
        except Exception as e:
            raw_all.append(str(e))
    # Majority vote
    if answers:
        from collections import Counter
        ans = Counter(answers).most_common(1)[0][0]
    else:
        ans = None
    return BaselineResult(answer=ans, raw='\n---\n'.join(raw_all), num_llm_calls=n,
                          tokens_estimated=900*n, time_s=time.time()-t0)

def baseline_only(client, problem: str) -> BaselineResult:
    """Single-agent baseline (no MAS scaffold)."""
    t0 = time.time()
    prompt = (f'You are a math expert. Solve the following problem step by step '
              f'and give the final numeric answer.\n\n{problem}')
    try:
        raw = _call(client, prompt, max_tokens=800)
        ans = _extract_last_number(raw)
        return BaselineResult(answer=ans, raw=raw, num_llm_calls=1,
                              tokens_estimated=900, time_s=time.time()-t0)
    except Exception as e:
        return BaselineResult(answer=None, raw=str(e), num_llm_calls=1,
                              tokens_estimated=0, time_s=time.time()-t0, error_type='exception')

BASELINE_FNS = {
    'b1_direct':        direct_answer,
    'b2_cot':           chain_of_thought,
    'b3_sc5':           lambda c, p: self_consistency(c, p, n=5),
    'b4_baseline_only': baseline_only,
}

def _row_from_baseline(system, preset, problem, gold_id, gold, br: BaselineResult):
    return {
        'problem_id': gold_id, 'system': system, 'preset': preset,
        'gold': gold, 'predicted': br.answer,
        'correct': _is_correct(br.answer, gold),
        'time_s': br.time_s, 'num_llm_calls': br.num_llm_calls,
        'tokens_estimated': br.tokens_estimated,
        'error_type': br.error_type,
        'timestamp': datetime.now().isoformat(),
    }

def _row_from_mas(system, preset, gold_id, gold, mas_out, time_s):
    pred_str = mas_out.get('mas', {}).get('answer', '')
    pred = _extract_last_number(str(pred_str))
    sht_block = mas_out.get('sht', {}) or {}
    siv_block = mas_out.get('siv', {}) or {}
    return {
        'problem_id': gold_id, 'system': system, 'preset': preset,
        'gold': gold, 'predicted': pred,
        'correct': _is_correct(pred, gold),
        'time_s': time_s,
        'num_llm_calls': sht_block.get('api_calls_used', 3),
        'tokens_estimated': 0,
        'error_type': '' if pred is not None else 'mas_returned_unknown',
        'sht_triggered': sht_block.get('triggered', False),
        'siv_invertible': siv_block.get('invertible'),
        'siv_verified': siv_block.get('verified'),
        'siv_execution_audit_passed': siv_block.get('execution_audit_passed'),
        'timestamp': datetime.now().isoformat(),
    }

# ------------------------------------------------------------------
# Run baselines (B1-B4)
# ------------------------------------------------------------------
if CONFIG['baselines_to_run']:
    bc = CONFIG['baseline_client']
    baseline_client = UnifiedLLMClient(provider=bc['provider'], model_override=bc['model'])
    for sys_name in CONFIG['baselines_to_run']:
        fn = BASELINE_FNS[sys_name]
        rows = _load_ckpt(sys_name)
        done_ids = {r['problem_id'] for r in rows}
        print(f'[{sys_name}] resuming with {len(done_ids)} done; total {len(PROBLEMS)}')
        budget_hit = False
        for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
            if p['problem_id'] in done_ids:
                continue
            try:
                br = fn(baseline_client, p['question'])
            except Exception as e:
                br = BaselineResult(answer=None, raw=traceback.format_exc()[-500:],
                                    num_llm_calls=0, tokens_estimated=0,
                                    time_s=0.0, error_type='exception')
            rows.append(_row_from_baseline(sys_name, bc['model'], p['question'],
                                           p['problem_id'], p['gold_answer'], br))
            if _budget_blocked(br.error_type):
                print(f'[{sys_name}] daily budget hit at problem {i}. Resume tomorrow.')
                budget_hit = True
            if (i + 1) % CONFIG['checkpoint_every'] == 0 or budget_hit:
                _save_ckpt(sys_name, rows)
            if budget_hit:
                break
            time.sleep(CONFIG['inter_problem_delay'])
        _save_ckpt(sys_name, rows)
        pd.DataFrame(rows).to_csv(_csv_path(sys_name), index=False)
        print(f'[{sys_name}] wrote {len(rows)} rows -> {_csv_path(sys_name)}')

# ------------------------------------------------------------------
# Run MAS variants (B5/B6/B7 and small-model presets)
# ------------------------------------------------------------------
for sys_name, preset, en_siv, en_sht in CONFIG['mas_variants']:
    rows = _load_ckpt(sys_name)
    done_ids = {r['problem_id'] for r in rows}
    print(f'[{sys_name}] preset={preset} siv={en_siv} sht={en_sht} '
          f'resuming with {len(done_ids)} done')
    pipeline = QualityAwarePipeline(
        heterogeneous_preset=preset, use_cache=False,
        enable_siv=en_siv, enable_sht=en_sht,
    )
    budget_hit = False
    for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
        if p['problem_id'] in done_ids:
            continue
        t0 = time.time()
        try:
            mas_out = pipeline.solver.solve(p['question'], str(p['gold_answer']))
        except Exception as e:
            print(f'  exception on {p["problem_id"]}: {e}')
            mas_out = {'mas': {'answer': 'unknown'}, 'siv': {}, 'sht': {}}
        elapsed = time.time() - t0
        rows.append(_row_from_mas(sys_name, preset, p['problem_id'],
                                  p['gold_answer'], mas_out, elapsed))
        if 'budget_exceeded' in str(mas_out).lower():
            print(f'[{sys_name}] daily budget hit at problem {i}. Resume tomorrow.')
            budget_hit = True
        if (i + 1) % CONFIG['checkpoint_every'] == 0 or budget_hit:
            _save_ckpt(sys_name, rows)
        if budget_hit:
            break
        time.sleep(CONFIG['inter_problem_delay'])
    _save_ckpt(sys_name, rows)
    pd.DataFrame(rows).to_csv(_csv_path(sys_name), index=False)
    print(f'[{sys_name}] wrote {len(rows)} rows -> {_csv_path(sys_name)}')

print('\nAll runs complete.')
print(token_budget.usage_report())

## Cell 4 — Aggregation

In [ ]:
# === Cell 4 — Aggregation ======================================================
# Glob all CSVs in RESULTS_DIR, dedupe (problem_id, system) keeping latest.

import glob
import pandas as pd

csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
print(f'Found {len(csv_paths)} CSVs in {RESULTS_DIR}')

frames = [pd.read_csv(p) for p in csv_paths]
if not frames:
    raise SystemExit('No results to aggregate. Run Cell 3 first.')
merged = pd.concat(frames, ignore_index=True)
merged['timestamp'] = pd.to_datetime(merged.get('timestamp'), errors='coerce')
merged = (merged.sort_values('timestamp')
                .drop_duplicates(['problem_id', 'system'], keep='last'))

results_dict = {sys_name: g.reset_index(drop=True)
                for sys_name, g in merged.groupby('system')}
summary = (merged.groupby('system')
                  .agg(n=('problem_id', 'nunique'),
                       accuracy=('correct', 'mean'),
                       avg_calls=('num_llm_calls', 'mean'),
                       avg_time=('time_s', 'mean'))
                  .sort_values('accuracy', ascending=False))
summary

## Cell 5 — Statistical Analysis

In [ ]:
# === Cell 5 — Statistics =======================================================
# McNemar tests + comparison table.

from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

REFERENCE = 'mas_sht_full'

def compute_all_metrics(results_dict, reference_system):
    rows = []
    for sys_name, df in results_dict.items():
        row = {
            'system': sys_name,
            'n': len(df),
            'accuracy': df['correct'].mean(),
            'avg_llm_calls': df['num_llm_calls'].mean(),
            'avg_time_s': df['time_s'].mean(),
        }
        if 'sht_triggered' in df.columns:
            row['sht_rate'] = df['sht_triggered'].mean()
        if 'siv_verified' in df.columns:
            row['siv_rate'] = df['siv_verified'].dropna().mean()
        rows.append(row)
    return pd.DataFrame(rows).sort_values('accuracy', ascending=False)

def run_mcnemar_tests(results_dict, reference_system):
    if reference_system not in results_dict:
        print(f'Reference {reference_system} not found.')
        return pd.DataFrame()
    ref_df = results_dict[reference_system].set_index('problem_id')
    rows = []
    for sys_name, df in results_dict.items():
        if sys_name == reference_system:
            continue
        cmp = df.set_index('problem_id').join(ref_df[['correct']], rsuffix='_ref')
        cmp = cmp.dropna(subset=['correct', 'correct_ref'])
        b = ((cmp['correct_ref'] == True) & (cmp['correct'] == False)).sum()
        c = ((cmp['correct_ref'] == False) & (cmp['correct'] == True)).sum()
        table = [[0, b], [c, 0]]
        try:
            result = mcnemar([[int(b+c > 0), int(b)], [int(c), 0]], exact=True)
            p = result.pvalue
        except Exception:
            p = float('nan')
        rows.append({'system': sys_name, 'reference': reference_system,
                     'b_ref_wins': int(b), 'c_sys_wins': int(c), 'p_value': p,
                     'significant': p < 0.05 if not np.isnan(p) else False})
    return pd.DataFrame(rows)

metrics_df = compute_all_metrics(results_dict, reference_system=REFERENCE)
mcnemar_df = run_mcnemar_tests(results_dict, reference_system=REFERENCE)

metrics_path = os.path.join(ARTIFACTS_DIR, 'comparison_table.csv')
metrics_df.to_csv(metrics_path, index=False)
mcnemar_df.to_csv(os.path.join(ARTIFACTS_DIR, 'mcnemar_results.csv'), index=False)

print('--- Per-system metrics ---')
print(metrics_df.to_string(index=False))
print()
print('--- McNemar (paired) ---')
print(mcnemar_df.to_string(index=False))

## Cell 6 — Plots

In [ ]:
# === Cell 6 — Plots ============================================================
# Three-panel figure: accuracy / avg LLM calls / McNemar significance.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import Image, display

def plot_comparison(metrics_df, mcnemar_df, output_path, reference_system, title_suffix=''):
    systems = metrics_df['system'].tolist()
    accuracies = metrics_df['accuracy'].tolist()
    avg_calls = metrics_df['avg_llm_calls'].tolist()

    sig_map = {}
    for _, row in mcnemar_df.iterrows():
        sig_map[row['system']] = row['significant']

    colors = ['#2ecc71' if s == reference_system else
              '#e74c3c' if sig_map.get(s, False) else '#3498db'
              for s in systems]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'MAS-SHT System Comparison {title_suffix}', fontsize=13, fontweight='bold')

    # Panel 1: Accuracy
    axes[0].barh(systems, accuracies, color=colors)
    axes[0].set_xlabel('Accuracy')
    axes[0].set_title('Accuracy by System')
    axes[0].set_xlim(0, 1)
    for i, v in enumerate(accuracies):
        axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

    # Panel 2: Avg LLM calls
    axes[1].barh(systems, avg_calls, color=colors)
    axes[1].set_xlabel('Avg LLM Calls')
    axes[1].set_title('Efficiency (LLM Calls)')
    for i, v in enumerate(avg_calls):
        axes[1].text(v + 0.05, i, f'{v:.1f}', va='center', fontsize=8)

    # Panel 3: McNemar p-values
    if not mcnemar_df.empty:
        mc_systems = mcnemar_df['system'].tolist()
        p_values = mcnemar_df['p_value'].tolist()
        mc_colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in p_values]
        axes[2].barh(mc_systems, [-np.log10(max(p, 1e-10)) for p in p_values], color=mc_colors)
        axes[2].axvline(-np.log10(0.05), color='black', linestyle='--', label='p=0.05')
        axes[2].set_xlabel('-log10(p-value)')
        axes[2].set_title(f'McNemar vs {reference_system}')
        axes[2].legend(fontsize=8)
    else:
        axes[2].text(0.5, 0.5, 'No McNemar data', ha='center', va='center')

    legend_patches = [
        mpatches.Patch(color='#2ecc71', label='Reference'),
        mpatches.Patch(color='#e74c3c', label='Sig. different (p<0.05)'),
        mpatches.Patch(color='#3498db', label='Not significant'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=8)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {output_path}')

fig_path = os.path.join(ARTIFACTS_DIR, 'comparison.png')
plot_comparison(
    metrics_df, mcnemar_df,
    output_path=fig_path,
    reference_system=REFERENCE,
    title_suffix=f'GSM8K-{DATASET_CONFIG["mode"]}-n{len(PROBLEMS)}'
)
display(Image(fig_path))